# 🚀 Chạy trên Google Colab
Chạy cell dưới đây để mount Google Drive và tự động cấu hình các thư mục dự án.

In [ ]:
# ===== MOUNT GOOGLE DRIVE =====
from google.colab import drive
drive.mount('/content/drive')

# ===== CÀI ĐẶT THƯ VIỆN & THIẾT LẬP ĐƯỜNG DẪN =====
!pip install -q underthesea

BASE_DIR = "/content/drive/MyDrive/TieuLuan_22130161"
DATA_DIR = f"{BASE_DIR}/data"
FN_DIR   = f"{BASE_DIR}/fn"
PKL_DIR  = f"{BASE_DIR}/pkl"
IMG_DIR  = f"{BASE_DIR}/image"

import os
os.makedirs(FN_DIR, exist_ok=True)
os.makedirs(PKL_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

print("✅ Môi trường Colab đã sẵn sàng!")


# Notebook 00 — Tiền xử lý Dữ liệu Phản hồi Sinh viên tiếng Việt

**Mục tiêu:** Làm sạch và chuẩn hóa toàn bộ bộ dữ liệu UIT-VSFC bao gồm:
1. Chuẩn hóa Unicode (NFC) và chuyển về chữ thường
2. Xóa ký tự lặp lại kéo dài (e.g. `dạaaaaa` → `dạa`)
3. Ánh xạ teencode và từ viết tắt qua từ điển 478 quy tắc
4. Loại bỏ dấu câu và ký hiệu đặc biệt
5. Tách từ tiếng Việt bằng thư viện `underthesea`

Kết quả được lưu ra thư mục `fn/` để các notebook tiếp theo sử dụng.

In [ ]:
# ===== CÀI ĐẶT THƯ VIỆN (chỉ chạy lần đầu) =====
# # !pip install pandas underthesea


In [ ]:
import os
import re
import unicodedata
import pandas as pd
from underthesea import word_tokenize

print("✅ Đã import thành công tất cả thư viện cần thiết!")
print(f"pandas version: {pd.__version__}")


## 1. Tải Từ điển Teencode

In [ ]:
# Đường dẫn tương đối từ thư mục notebooks/ lên thư mục gốc của dự án
DATA_DIR = f"{BASE_DIR}/data"
FN_DIR = f"{BASE_DIR}/fn"
os.makedirs(FN_DIR, exist_ok=True)

def load_teencode_dict(file_path):
    """Tải từ điển ánh xạ teencode & viết tắt từ file .txt (định dạng tab-separated)."""
    teencode_dict = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    teencode_dict[parts[0].strip()] = parts[1].strip()
        print(f"Đã tải {len(teencode_dict)} quy tắc từ điển teencode!")
    except FileNotFoundError:
        print(f"Không tìm thấy file từ điển: {file_path}")
    return teencode_dict

TEENCODE_DICT = load_teencode_dict(os.path.join(DATA_DIR, "teencode.txt"))

# Xem thử vài ví dụ
sample = list(TEENCODE_DICT.items())[:5]
for k, v in sample:
    print(f"  '{k}' → '{v}'")


## 2. Định nghĩa Hàm Tiền xử lý Lõi

In [ ]:
def normalize_teencode(text, teencode_dict=TEENCODE_DICT):
    """Thay thế từng từ bằng cách tra từ điển teencode."""
    words = str(text).split()
    return " ".join([teencode_dict.get(w, w) for w in words])

def clean_vietnamese_text(text):
    """
    Hàm tiền xử lý chính. Thực hiện tuần tự các bước:
    1. Xử lý giá trị rỗng (NaN)
    2. Chuẩn hóa Unicode NFC, chuyển về chữ thường
    3. Xóa ký tự lặp lại liên tiếp > 2 lần
    4. Ánh xạ teencode qua từ điển
    5. Loại bỏ dấu câu & ký hiệu đặc biệt
    6. Thu gọn khoảng trắng thừa
    7. Tách từ tiếng Việt bằng underthesea
    """
    if pd.isna(text) or str(text).strip() == "":
        return ""
    # Bước 2
    text = unicodedata.normalize("NFC", str(text)).lower()
    # Bước 3
    text = re.sub(r'([a-zà-ỹđ])\1{2,}', r'\1\1', text)
    # Bước 4
    text = normalize_teencode(text)
    # Bước 5
    text = re.sub(r'[^\w\s]', ' ', text)
    # Bước 6
    text = re.sub(r'\s+', ' ', text).strip()
    # Bước 7 — tách từ (e.g. "sinh viên" → "sinh_viên")
    if text:
        text = word_tokenize(text, format="text")
    return text

# === Kiểm thử nhanh ===
tests = [
    "Thầy dạy hay lắm nhưng bài tập nhìu quáaaaaa :))",
    "ko hỉu j hết, mk chán lắm r",
    "okela thầy giảng dc đó nha, thx thầy nhìu"
]
print("=== VÍ DỤ TIỀN XỬ LÝ ===")
for t in tests:
    print(f"  Gốc  : {t}")
    print(f"  Sạch : {clean_vietnamese_text(t)}")
    print()


## 3. Xử lý Hàng loạt Toàn bộ Dataset

In [ ]:
datasets = ['train.csv', 'validation.csv', 'test.csv']

print("=" * 60)
print("BẮT ĐẦU TIỀN XỬ LÝ TOÀN BỘ DỮ LIỆU UIT-VSFC")
print("=" * 60)

for file_name in datasets:
    input_path = os.path.join(DATA_DIR, file_name)
    output_path = os.path.join(FN_DIR, f"cleaned_{file_name}")

    print(f"\n⌛ Đang xử lý: {file_name}...")
    df = pd.read_csv(input_path)
    df['comment_cleaned'] = df['comment'].apply(clean_vietnamese_text)
    df['comment_cleaned'] = df['comment_cleaned'].fillna('')
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Xong! Đã lưu {len(df)} mẫu → {output_path}")

print("\n" + "=" * 60)
print("TIỀN XỬ LÝ HOÀN TẤT!")
print("=" * 60)


## 4. Thống kê Kết quả sau Tiền xử lý

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Đọc lại kết quả đã lưu
train_df = pd.read_csv(os.path.join(FN_DIR, "cleaned_train.csv")).fillna('')
val_df   = pd.read_csv(os.path.join(FN_DIR, "cleaned_validation.csv")).fillna('')
test_df  = pd.read_csv(os.path.join(FN_DIR, "cleaned_test.csv")).fillna('')

print("THỐNG KÊ BỘ DỮ LIỆU UIT-VSFC SAU TIỀN XỬ LÝ")
print("-" * 50)
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    counts = df['label'].value_counts().sort_index()
    total  = len(df)
    print(f"\n{name} ({total} mẫu):")
    label_map = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
    for lbl, cnt in counts.items():
        print(f"  Lớp {lbl} ({label_map[lbl]}): {cnt:5d} mẫu ({cnt/total*100:.1f}%)")

# Vẽ biểu đồ phân bố nhãn
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
label_names = ['Tieu cuc (0)', 'Trung tinh (1)', 'Tich cuc (2)']
colors = ['#d73027', '#fee08b', '#1a9850']

for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    counts = df['label'].value_counts().sort_index()
    ax.bar(label_names, counts.values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'Phan bo nhan - {name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('So luong mau')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 30, str(v), ha='center', fontweight='bold')

plt.tight_layout()
os.makedirs(IMG_DIR, exist_ok=True)
plt.savefig(os.path.join(IMG_DIR, "data_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu biểu đồ phân bố nhãn → ../image/data_distribution.png")
